# Deep Residual Learning for Image Recognition — CIFAR-10 reproduction

**ArXivist-generated reproduction notebook**
Paper: [arXiv:1512.03385](https://arxiv.org/abs/1512.03385) (He, Zhang, Ren, Sun — Microsoft Research, December 2015)
Generated: 2026-05-19

This notebook:
1. Verifies your environment and installs the package.
2. Walks through the model components (`BasicBlock`, `IdentityPadShortcut`, `ResNetCIFAR`) with toy inputs.
3. Runs a **mini-training** on synthetic data to confirm the loop converges — no CIFAR-10 download needed.
4. Shows the paper's reported CIFAR-10 results and points at the full training script.

Runs end-to-end in ~1 minute on CPU.

## 1. Environment check

In [ ]:
import sys
import torch

print(f"Python:        {sys.version.split()[0]}")
print(f"PyTorch:       {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:           {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM:          {vram_gb:.1f} GB")
else:
    print("Running on CPU — mini-training in this notebook is fine; full training is slower.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Selected device: {device}")

## 2. Install the project (editable)

This makes `import resnet_cifar` work from anywhere. The notebook lives in `notebooks/`, so the package root is one directory up.

In [ ]:
import subprocess
from pathlib import Path

repo_root = Path.cwd().parent
print(f"Installing from: {repo_root}")

try:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", str(repo_root)],
        capture_output=True, text=True, timeout=300,
    )
    if result.returncode != 0:
        print("Install failed — falling back to sys.path injection.")
        print(result.stderr[-1000:])
        sys.path.insert(0, str(repo_root / "src"))
    else:
        print("Install OK.")
except Exception as e:
    print(f"Install raised {e!r} — falling back to sys.path injection.")
    sys.path.insert(0, str(repo_root / "src"))

## 3. Paper overview

The paper introduces **residual learning** to make very deep networks trainable. Each block learns a residual $\mathcal{F}(x, \{W_i\})$ that gets added to its input via a shortcut connection:

$$y = \mathcal{F}(x, \{W_i\}) + x \qquad (\text{Eq. 1})$$

For the CIFAR-10 experiments (paper Section 4.2), the architecture is simple and parameter-efficient:

- 32×32 RGB input.
- A 3×3 conv with 16 filters, then BN + ReLU.
- Three stages of `n` `BasicBlock`s with feature-map sizes 32, 16, 8 and filter counts 16, 32, 64.
- Stride-2 downsampling on the first block of each non-initial stage.
- Identity shortcuts ("Option A") with zero-padded extra channels when dimensions change.
- Global average pool → Linear(64→10) → softmax.

Depth = $6n + 2$. We support $n \in \{3, 5, 7, 9, 18\}$ giving ResNet-{20, 32, 44, 56, 110}.

## 4. Components
### 4a. `IdentityPadShortcut` (Option A, paper Sec. 3.3 + 4.2)

Parameter-free shortcut for dimension-changing blocks: stride-2 slice + zero-padded extra channels.

In [ ]:
from resnet_cifar.models import IdentityPadShortcut

shortcut = IdentityPadShortcut(in_planes=16, out_planes=32, stride=2)
x = torch.randn(2, 16, 32, 32)
y = shortcut(x)
print(f"Shortcut:      {shortcut}")
print(f"Input shape:   {tuple(x.shape)}  (expected [B,16,32,32])")
print(f"Output shape:  {tuple(y.shape)}  (expected [B,32,16,16])")
assert y.shape == (2, 32, 16, 16)
# The first 16 channels are the downsampled identity; the last 16 are zero.
print(f"New channels are zero: {torch.all(y[:, 16:] == 0).item()}")

### 4b. `BasicBlock` (paper Eq. 1)

Two 3×3 convs with BN, residual addition, then ReLU.
$$y = \sigma\big(\mathcal{F}(x,\{W_i\}) + x\big), \quad \mathcal{F} = W_2\,\sigma(W_1 x)$$

In [ ]:
from resnet_cifar.models import BasicBlock

# Same-resolution block (identity shortcut)
block_same = BasicBlock(in_planes=16, planes=16, stride=1)
x = torch.randn(2, 16, 32, 32)
y = block_same(x)
print(f"Block (same):  {block_same}")
print(f"  in {tuple(x.shape)} -> out {tuple(y.shape)} (expected [2,16,32,32])")

# Downsampling block (Option A shortcut)
block_down = BasicBlock(in_planes=16, planes=32, stride=2, shortcut_option='A')
y = block_down(x)
print(f"Block (down):  {block_down}")
print(f"  in {tuple(x.shape)} -> out {tuple(y.shape)} (expected [2,32,16,16])")

### 4c. Full network: `ResNetCIFAR`

Three stages, each with `n` BasicBlocks. Below we instantiate ResNet-20 ($n=3$) and check its parameter count against the paper's quote ("about 0.27M").

In [ ]:
from resnet_cifar.models import build_model

for name in ["resnet20", "resnet32", "resnet44", "resnet56", "resnet110"]:
    m = build_model(name)
    print(f"{name:>10}  depth={m.depth:>3}  params={m.num_parameters():>9,d}")

model = build_model("resnet20").to(device)
x = torch.randn(4, 3, 32, 32, device=device)
logits = model(x)
print(f"\nForward pass: in {tuple(x.shape)} -> out {tuple(logits.shape)}  (expected [4, 10])")

## 5. Mini-training on synthetic data

Three classes of synthetic 32×32 images with class-conditional means — enough structure that loss should drop visibly over ~30 SGD steps. No downloads, no internet.

In [ ]:
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0)
n_per_class, n_classes = 60, 3
class_means = torch.randn(n_classes, 3, 32, 32) * 0.3
images, labels = [], []
for c in range(n_classes):
    images.append(class_means[c].unsqueeze(0).expand(n_per_class, -1, -1, -1) + 0.2 * torch.randn(n_per_class, 3, 32, 32))
    labels.append(torch.full((n_per_class,), c, dtype=torch.long))
images = torch.cat(images, dim=0)
labels = torch.cat(labels, dim=0)
ds = TensorDataset(images, labels)
loader = DataLoader(ds, batch_size=32, shuffle=True)
print(f"Synthetic dataset: {len(ds)} samples, {n_classes} classes, image shape {tuple(images[0].shape)}")

In [ ]:
mini_model = build_model("resnet20", num_classes=n_classes).to(device)
optimizer = torch.optim.SGD(mini_model.parameters(), lr=0.05, momentum=0.9, weight_decay=1e-4)

print(f"Mini model: {mini_model}")

mini_model.train()
history = []
for step, (x, y) in enumerate(loader):
    if step >= 30:
        break
    x = x.to(device); y = y.to(device)
    optimizer.zero_grad()
    logits = mini_model(x)
    loss = F.cross_entropy(logits, y)
    loss.backward()
    optimizer.step()
    history.append(float(loss.item()))
    if step % 5 == 0:
        print(f"  step {step:>2}  loss={loss.item():.4f}")

print(f"\nLoss start: {history[0]:.4f}")
print(f"Loss end:   {history[-1]:.4f}")
if history[-1] < history[0]:
    print("✓ Loss is decreasing — training loop wired up correctly.")
else:
    print("✗ Loss did not decrease — something is wrong.")

In [ ]:
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6, 3.5))
    plt.plot(history, marker='o', markersize=3)
    plt.xlabel("step"); plt.ylabel("loss")
    plt.title("Mini-training loss (synthetic 3-class data)")
    plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib not installed — skipping plot.")
    print(history)

## 6. Paper's reported CIFAR-10 results

From Table 6 of the paper (test error, %):

In [ ]:
paper_results = {
    "resnet20":  {"test_error_percent": 8.75, "note": ""},
    "resnet32":  {"test_error_percent": 7.51, "note": ""},
    "resnet44":  {"test_error_percent": 7.17, "note": ""},
    "resnet56":  {"test_error_percent": 6.97, "note": ""},
    "resnet110": {"test_error_percent": 6.43, "note": "median of 5 runs; mean 6.61 ± 0.16"},
}
print(f"{'model':>10}  {'test_err%':>10}  notes")
for name, r in paper_results.items():
    print(f"{name:>10}  {r['test_error_percent']:>10.2f}  {r['note']}")

## 7. Full training (separate process)

The mini-training above runs in ~1 minute and only verifies the wiring. To **actually reproduce** the paper's results, run from a terminal:

```bash
# ~15 min on RTX 3090, ~1 hour on a modern CPU
python train.py --config configs/resnet20.yaml --output-dir runs

# Or ResNet-110 (~2 hours on GPU)
python train.py --config configs/resnet110.yaml --output-dir runs
```

Then evaluate the saved best checkpoint:

```bash
python evaluate.py --config configs/resnet20.yaml --checkpoint runs/resnet20/best.pt
```

## 8. Implementation notes (from the SIR)

Things the paper didn't fully nail down; their resolutions are exposed as config flags:

1. **Mean subtraction**: per-pixel (default, paper wording) vs per-channel. Toggle via `data.mean_subtraction`.
2. **Weight decay on BN params**: excluded by default (modern convention). Toggle via `training.apply_wd_to_bn`.
3. **ResNet-110 LR warmup**: auto-enabled when `model.name == 'resnet110'`. The model otherwise fails to converge.

See `arxivist-workspace/sir-registry/arxiv_1512_003385/sir.json` → `implementation_assumptions[]` and `ambiguities[]` for the complete record.